
# Hybrid Product Search and Retrieval Evaluation

A product-search experiment combining structured SQL filters
with semantic vector ranking.

The system compares vector-only retrieval with a hybrid
approach that enforces customer requirements before ranking.

**Technologies:** Python, SQLite, SentenceTransformers,
NumPy, and Pandas.

**Course references:**
- [Hybrid Retrieval with SQL Filtering and Embeddings](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/introduction_to_hybrid_retrieval_with_sql_filtering_%26_embeddings.ipynb)
- [How to Evaluate Retrievers](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/how_to_evaluate_retrievers.ipynb)

The dataset and evaluation cases are custom educational
examples inspired by the course. Embeddings run locally,
and structured search filters are specified manually.


In [1]:

%pip install -q sentence-transformers numpy pandas

import sqlite3
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer



## 1. Product Catalogue

Create a synthetic catalogue of audio products with
descriptions, categories, prices, stock availability,
wireless connectivity, and noise cancellation.

These attributes allow the retrieval system to enforce
explicit customer requirements.


In [2]:

conn = sqlite3.connect(":memory:")

conn.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL CHECK (price >= 0),
    in_stock INTEGER NOT NULL,
    wireless INTEGER NOT NULL,
    noise_cancelling INTEGER NOT NULL,
    description TEXT NOT NULL,
    embedding BLOB
)
""")

products = [
    (1, "Everyday Wireless Earbuds", "earbuds", 29.99, 1, 1, 0,
     "Compact Bluetooth earbuds for music and daily commuting"),

    (2, "Sport Wireless Earbuds", "earbuds", 44.99, 1, 1, 0,
     "Secure fit Bluetooth earbuds for running and exercise"),

    (3, "ANC Wireless Earbuds", "earbuds", 89.99, 1, 1, 1,
     "Wireless in-ear earbuds with active noise cancellation"),

    (4, "Travel ANC Earbuds", "earbuds", 69.99, 0, 1, 1,
     "Bluetooth earbuds with active noise cancellation for travel"),

    (5, "Basic Wired Earbuds", "earbuds", 14.99, 1, 0, 0,
     "Wired in-ear earbuds with a standard audio connector"),

    (6, "Everyday Wireless Headphones", "headphones", 74.99, 1, 1, 0,
     "Wireless over-ear Bluetooth headphones for daily listening"),

    (7, "Premium ANC Headphones", "headphones", 129.99, 1, 1, 1,
     "Over-ear wireless headphones with active noise cancellation"),

    (8, "Wired Studio Headphones", "headphones", 54.99, 1, 0, 0,
     "Wired over-ear headphones for recording and monitoring"),

    (9, "Budget Wireless Headphones", "headphones", 49.99, 1, 1, 0,
     "Affordable Bluetooth over-ear headphones for music"),

    (10, "Wired Gaming Headset", "headphones", 59.99, 1, 0, 0,
     "Wired gaming headset with microphone and padded ear cups"),

    (11, "Travel Noise Cancelling Headphones", "headphones",
     199.99, 1, 1, 1,
     "Premium wireless over-ear headphones with noise cancellation"),

    (12, "Kids Wireless Headphones", "headphones", 39.99, 0, 1, 0,
     "Bluetooth headphones with volume limiting for children"),

    (13, "Compact Bluetooth Speaker", "speakers", 34.99, 1, 1, 0,
     "Small portable wireless speaker for indoor and outdoor music"),

    (14, "Waterproof Bluetooth Speaker", "speakers", 59.99, 1, 1, 0,
     "Portable waterproof Bluetooth speaker for travel and outdoors"),

    (15, "Home Bluetooth Speaker", "speakers", 109.99, 1, 1, 0,
     "Large wireless speaker for home audio and music"),

    (16, "Smart WiFi Speaker", "speakers", 89.99, 1, 0, 0,
     "Smart home speaker with WiFi streaming and voice controls"),

    (17, "TV Soundbar", "speakers", 179.99, 1, 0, 0,
     "Home theatre soundbar for television audio"),

    (18, "Earbud Charging Case", "accessories", 19.99, 1, 0, 0,
     "Replacement charging case for compatible wireless earbuds"),

    (19, "Headphone Carry Case", "accessories", 15.99, 1, 0, 0,
     "Protective carrying case for over-ear headphones"),

    (20, "AUX Audio Cable", "accessories", 8.99, 1, 0, 0,
     "Standard wired audio cable for connecting audio devices"),

    (21, "USB-C Audio Adapter", "accessories", 11.99, 1, 0, 0,
     "USB-C adapter for connecting wired headphones"),

    (22, "Replacement Ear Tips", "accessories", 7.99, 1, 0, 0,
     "Silicone replacement tips for in-ear earbuds"),

    (23, "Bluetooth Audio Transmitter", "accessories",
     26.99, 1, 1, 0,
     "Wireless Bluetooth transmitter for compatible audio devices"),

    (24, "Replacement Ear Cushions", "accessories",
     13.99, 1, 0, 0,
     "Soft replacement cushions for over-ear headphones")
]

conn.executemany("""
INSERT INTO products
(id, name, category, price, in_stock, wireless,
 noise_cancelling, description)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", products)

conn.commit()

catalogue = pd.read_sql_query("""
SELECT id, name, category, price, in_stock,
       wireless, noise_cancelling
FROM products
ORDER BY category, price
""", conn)

display(catalogue)

print("Number of products:", len(catalogue))


,id,name,category,price,in_stock,wireless,noise_cancelling
0,22,Replacement Ear Tips,accessories,7.99,1,0,0
1,20,AUX Audio Cable,accessories,8.99,1,0,0
2,21,USB-C Audio Adapter,accessories,11.99,1,0,0
3,24,Replacement Ear Cushions,accessories,13.99,1,0,0
4,19,Headphone Carry Case,accessories,15.99,1,0,0
5,18,Earbud Charging Case,accessories,19.99,1,0,0
6,23,Bluetooth Audio Transmitter,accessories,26.99,1,1,0
7,5,Basic Wired Earbuds,earbuds,14.99,1,0,0
8,1,Everyday Wireless Earbuds,earbuds,29.99,1,1,0
9,2,Sport Wireless Earbuds,earbuds,44.99,1,1,0


Number of products: 24



## 2. Semantic Embeddings

Generate product embeddings from names and descriptions
using a local SentenceTransformer model.

Store the embeddings in SQLite for similarity search.


In [3]:

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

rows = conn.execute("""
SELECT id, name, description
FROM products
ORDER BY id
""").fetchall()

texts = [
    f"{name}. {description}"
    for _, name, description in rows
]

vectors = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True
)

for (product_id, _, _), vector in zip(rows, vectors):
    conn.execute(
        "UPDATE products SET embedding = ? WHERE id = ?",
        (
            vector.astype(np.float32).tobytes(),
            product_id
        )
    )

conn.commit()

print("Products embedded:", len(vectors))
print("Vector dimensions:", vectors.shape[1])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Products embedded: 24
Vector dimensions: 384



## 3. Vector and Hybrid Retrieval

**Vector search:** Rank all products using embedding similarity.

**Hybrid search:** Apply SQL constraints first, then
rank the eligible products by embedding similarity.

Both methods use the same embedding model.


In [4]:

def search_products(
    semantic_query,
    filters=None,
    top_k=3
):
    sql = """
        SELECT id, name, category, price,
               in_stock, wireless, noise_cancelling,
               embedding
        FROM products
        WHERE 1=1
    """

    parameters = []

    # Apply structured filters if provided
    if filters:
        allowed_filters = {
            "category": "category",
            "max_price": "price",
            "in_stock": "in_stock",
            "wireless": "wireless",
            "noise_cancelling": "noise_cancelling"
        }

        if set(filters) - set(allowed_filters):
            raise ValueError("Unsupported search filter")

        for field, value in filters.items():
            if field == "max_price":
                sql += " AND price <= ?"
            else:
                sql += f" AND {allowed_filters[field]} = ?"

            parameters.append(value)

    candidates = conn.execute(
        sql, parameters
    ).fetchall()

    if not candidates:
        return []

    # Embed the search query
    query_vector = embedding_model.encode(
        semantic_query,
        normalize_embeddings=True
    )

    results = []

    for row in candidates:
        (
            product_id, name, category, price,
            in_stock, wireless, noise_cancelling,
            embedding_blob
        ) = row

        product_vector = np.frombuffer(
            embedding_blob,
            dtype=np.float32
        )

        # Both vectors are normalised, so dot product
        # equals cosine similarity
        score = float(
            np.dot(query_vector, product_vector)
        )

        results.append({
            "id": product_id,
            "name": name,
            "category": category,
            "price": price,
            "in_stock": in_stock,
            "wireless": wireless,
            "noise_cancelling": noise_cancelling,
            "similarity": score
        })

    results.sort(
        key=lambda item: (-item["similarity"], item["id"])
    )

    return results[:top_k]


In [5]:

customer_request = (
    "I want wireless earbuds for everyday music "
    "under $50, and they must be in stock."
)

semantic_query = "wireless earbuds for everyday music"

filters = {
    "category": "earbuds",
    "max_price": 50,
    "in_stock": 1,
    "wireless": 1
}

vector_results = search_products(
    semantic_query,
    filters=None,
    top_k=3
)

hybrid_results = search_products(
    semantic_query,
    filters=filters,
    top_k=3
)

print("Customer request:", customer_request)

print("\nVECTOR SEARCH:")
display(pd.DataFrame(vector_results))

print("\nHYBRID SEARCH:")
display(pd.DataFrame(hybrid_results))


Customer request: I want wireless earbuds for everyday music under $50, and they must be in stock.

VECTOR SEARCH:


,id,name,category,price,in_stock,wireless,noise_cancelling,similarity
0,1,Everyday Wireless Earbuds,earbuds,29.99,1,1,0,0.860748
1,6,Everyday Wireless Headphones,headphones,74.99,1,1,0,0.751304
2,9,Budget Wireless Headphones,headphones,49.99,1,1,0,0.725572



HYBRID SEARCH:


,id,name,category,price,in_stock,wireless,noise_cancelling,similarity
0,1,Everyday Wireless Earbuds,earbuds,29.99,1,1,0,0.860748
1,2,Sport Wireless Earbuds,earbuds,44.99,1,1,0,0.683700


In [6]:

request = (
    "Wireless noise-cancelling over-ear headphones "
    "under $80, in stock."
)

results = search_products(
    semantic_query="wireless noise cancelling over ear headphones",
    filters={
        "category": "headphones",
        "max_price": 80,
        "in_stock": 1,
        "wireless": 1,
        "noise_cancelling": 1
    },
    top_k=3
)

print("Customer request:", request)

if not results:
    print("No products satisfy all requirements.")
else:
    display(pd.DataFrame(results))


Customer request: Wireless noise-cancelling over-ear headphones under $80, in stock.
No products satisfy all requirements.



## 4. Retrieval Evaluation

Evaluate both retrieval methods using four product-search
scenarios with manually defined relevance labels.

A product is relevant only if it matches the search intent
and all required constraints.

Metrics:
- Precision@3
- Recall@3
- Mean Reciprocal Rank (MRR)
- NDCG@3
- Constraint Satisfaction Rate (CSR)
- Returned Precision

A separate test checks how the system handles requests
with no eligible products.


In [7]:

test_cases = [
    {
        "request": "Wireless earbuds under $50 for everyday listening",
        "semantic_query": "wireless earbuds for everyday music",
        "filters": {
            "category": "earbuds",
            "max_price": 50,
            "in_stock": 1,
            "wireless": 1
        },
        "relevant_ids": {1, 2}
    },
    {
        "request": "Wireless over-ear headphones under $80",
        "semantic_query": "wireless over ear headphones for music",
        "filters": {
            "category": "headphones",
            "max_price": 80,
            "in_stock": 1,
            "wireless": 1
        },
        "relevant_ids": {6, 9}
    },
    {
        "request": "Portable Bluetooth speakers under $70",
        "semantic_query": "portable Bluetooth speaker for music",
        "filters": {
            "category": "speakers",
            "max_price": 70,
            "in_stock": 1,
            "wireless": 1
        },
        "relevant_ids": {13, 14}
    },
    {
        "request": "Noise cancelling wireless earbuds under $100",
        "semantic_query": "wireless earbuds with active noise cancellation",
        "filters": {
            "category": "earbuds",
            "max_price": 100,
            "in_stock": 1,
            "wireless": 1,
            "noise_cancelling": 1
        },
        "relevant_ids": {3}
    }
]


In [16]:

def meets_constraints(product, filters):
    for field, value in filters.items():
        if field == "max_price":
            if product["price"] > value:
                return False
        elif product[field] != value:
            return False

    return True


def evaluate_results(results, relevant_ids, filters, k=3):
    ids = [item["id"] for item in results[:k]]

    hits = sum(
        product_id in relevant_ids
        for product_id in ids
    )

    precision = hits / k

    # Percentage of returned products that are relevant.
    # Unlike Precision@K, this handles fewer than K results.
    returned_precision = (
        hits / len(ids) if ids else None
    )

    recall = (
        hits / len(relevant_ids)
        if relevant_ids else 0.0
    )

    reciprocal_rank = next(
        (
            1 / rank
            for rank, product_id in enumerate(ids, start=1)
            if product_id in relevant_ids
        ),
        0.0
    )

    def dcg(relevance):
        return sum(
            (2**rel - 1) / np.log2(rank + 2)
            for rank, rel in enumerate(relevance)
        )

    actual_relevance = [
        int(product_id in relevant_ids)
        for product_id in ids
    ]

    ideal_relevance = [
        1
    ] * min(len(relevant_ids), k)

    ideal_dcg = dcg(ideal_relevance)

    ndcg = (
        dcg(actual_relevance) / ideal_dcg
        if ideal_dcg > 0 else 0.0
    )

    csr = (
        sum(meets_constraints(p, filters) for p in results)
        / len(results)
        if results else None
    )

    return {
        "Precision@3": precision,
        "Returned Precision": returned_precision,
        "Recall@3": recall,
        "RR": reciprocal_rank,
        "NDCG@3": ndcg,
        "CSR": csr
    }

In [17]:

all_products = {
    row["id"]: row
    for row in search_products(
        "audio equipment",
        top_k=len(products)
    )
}

for case in test_cases:
    for product_id in case["relevant_ids"]:
        assert product_id in all_products

        assert meets_constraints(
            all_products[product_id],
            case["filters"]
        ), f"Invalid relevance label: {product_id}"

print("All relevance labels satisfy their required constraints.")


All relevance labels satisfy their required constraints.


In [18]:

evaluation_rows = []

for case in test_cases:
    for method, filters in [
        ("Vector", None),
        ("Hybrid", case["filters"])
    ]:
        results = search_products(
            semantic_query=case["semantic_query"],
            filters=filters,
            top_k=3
        )

        metrics = evaluate_results(
            results,
            case["relevant_ids"],
            case["filters"],
            k=3
        )

        evaluation_rows.append({
            "Query": case["request"],
            "Method": method,
            "Retrieved IDs": [p["id"] for p in results],
            **metrics
        })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

summary = (
    evaluation_df
    .groupby("Method")[
        [
            "Precision@3",
            "Returned Precision",
            "Recall@3",
            "RR",
            "NDCG@3",
            "CSR"
        ]
    ]
    .mean()
    .rename(columns={"RR": "MRR"})
)

print("\nAverage evaluation metrics:")
display(summary.round(4))


,Query,Method,Retrieved IDs,Precision@3,Returned Precision,Recall@3,RR,NDCG@3,CSR
0,Wireless earbuds under $50 for everyday listening,Vector,"[1, 6, 9]",0.333333,0.333333,0.5,1.0,0.613147,0.333333
1,Wireless earbuds under $50 for everyday listening,Hybrid,"[1, 2]",0.666667,1.000000,1.0,1.0,1.000000,1.000000
2,Wireless over-ear headphones under $80,Vector,"[9, 6, 11]",0.666667,0.666667,1.0,1.0,1.000000,0.666667
3,Wireless over-ear headphones under $80,Hybrid,"[9, 6]",0.666667,1.000000,1.0,1.0,1.000000,1.000000
4,Portable Bluetooth speakers under $70,Vector,"[13, 15, 14]",0.666667,0.666667,1.0,1.0,0.919721,0.666667
5,Portable Bluetooth speakers under $70,Hybrid,"[13, 14]",0.666667,1.000000,1.0,1.0,1.000000,1.000000
6,Noise cancelling wireless earbuds under $100,Vector,"[3, 7, 11]",0.333333,0.333333,1.0,1.0,1.000000,0.333333
7,Noise cancelling wireless earbuds under $100,Hybrid,[3],0.333333,1.000000,1.0,1.0,1.000000,1.000000



Average evaluation metrics:


,Precision@3,Returned Precision,Recall@3,MRR,NDCG@3,CSR
Method,,,,,,
Hybrid,0.5833,1.0,1.000,1.0,1.0000,1.0
Vector,0.5000,0.5,0.875,1.0,0.8832,0.5


In [19]:

# Every hybrid result must satisfy its filters
for case in test_cases:
    results = search_products(
        case["semantic_query"],
        filters=case["filters"]
    )

    assert all(
        meets_constraints(p, case["filters"])
        for p in results
    )

# No-match scenario must return an empty list
no_match = search_products(
    "wireless noise cancelling over ear headphones",
    filters={
        "category": "headphones",
        "max_price": 80,
        "in_stock": 1,
        "wireless": 1,
        "noise_cancelling": 1
    }
)

assert no_match == []

# The database must contain the intended catalogue
assert conn.execute(
    "SELECT COUNT(*) FROM products"
).fetchone()[0] == 24

print("All validation checks passed.")


All validation checks passed.



## 5. Findings and Limitations

The experiment compared vector-only retrieval with
SQL-filtered semantic search across four test queries.

### Results

| Metric | Vector | Hybrid |
|---|---:|---:|
| Precision@3 | 0.5000 | 0.5833 |
| Returned Precision | 0.5000 | 1.0000 |
| Recall@3 | 0.8750 | 1.0000 |
| MRR | 1.0000 | 1.0000 |
| NDCG@3 | 0.8832 | 1.0000 |
| Constraint Satisfaction Rate | 0.5000 | 1.0000 |

Hybrid retrieval enforced all specified product constraints
and retrieved all manually labelled relevant products.

Both methods achieved an MRR of 1.0, meaning the first
retrieved product was relevant for every evaluated query.

The no-match test confirmed that hybrid search returns
an empty result when no product satisfies all requirements.

### Limitations

- Synthetic catalogue of 24 audio products
- Four manually labelled evaluation queries
- Manually supplied filters instead of LLM-based extraction
- Small dataset with simple, explicit eligibility conditions
- Local embedding model and no LLM answer generation

These results apply only to this controlled experiment.
They do not establish general superiority of hybrid retrieval.
